# GCP Vertex AI TPM 할당량 소모 방식 검증 핸즈온

## 목적

이 노트북은 GCP Vertex AI 환경에서 **TPM(Tokens Per Minute) 할당량이 실제로 어떻게 차감되는지** 직접 검증하기 위한 테스트입니다.

특히, **캐시된 토큰(Cached Tokens)이 TPM 할당량에서 할인 없이 1:1로 차감되는지** 여부를 확인합니다.

### 검증 결과의 활용

이 테스트를 통해 확인한 `usageMetadata`의 정확한 토큰 카운트 구조는, 향후 사내 개발자들을 위해 **Cloud Run에 배포할 게이트웨이 프록시(예: LiteLLM)**에서 트래픽을 통제하고 **BigQuery로 토큰 사용량 로그를 적재**할 때 할당량 제한 로직을 설계하는 핵심 기준이 됩니다.

### 테스트 흐름

| 단계 | 내용 | 방법 |
|------|------|------|
| **1단계** | 12k 캐시 생성 및 다중 호출 테스트 | Python 코드 실행 |
| **2단계** | GCP 콘솔에서 TPM 차감 그래프 확인 | 콘솔 모니터링 |

---
## 사전 준비

### 필수 패키지 설치

Vertex AI SDK가 설치되어 있지 않은 경우 아래 셀을 실행하세요.

In [ ]:
!pip install --upgrade google-cloud-aiplatform

### GCP 인증

로컬 환경에서 실행하는 경우, 아래 셀로 인증합니다.  
Vertex AI Workbench나 Cloud Shell에서 실행 중이라면 이 단계를 건너뛰세요.

In [ ]:
# 로컬 환경에서만 실행 (Workbench/Cloud Shell에서는 불필요)
# from google.colab import auth
# auth.authenticate_user()

# 또는 gcloud CLI 인증
# !gcloud auth application-default login

---
## 1단계: Python 코드로 12k 캐시 생성 및 다중 호출 테스트

게이트웨이를 거치지 않고 **Vertex AI SDK를 직접 호출**하여 순수한 베이스라인 지표를 발생시킵니다.

### 1-1. 환경 변수 설정

> **아래 `PROJECT_ID`를 본인 GCP 프로젝트 ID로 반드시 수정하세요.**

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"  # ← 본인 프로젝트 ID로 변경
LOCATION = "us-central1"
MODEL_ID = "gemini-2.5-flash"  # Gemini 2.5 Flash

# 테스트 파라미터
NUM_CALLS = 10           # API 호출 횟수
CACHE_TTL_HOURS = 1      # 캐시 유지 시간
SYSTEM_PROMPT_REPEAT = 500  # 시스템 프롬프트 반복 횟수 (약 10k~12k 토큰 생성용)

print(f"프로젝트: {PROJECT_ID}")
print(f"리전: {LOCATION}")
print(f"모델: {MODEL_ID}")
print(f"호출 횟수: {NUM_CALLS}회")

### 1-2. Vertex AI 초기화

In [ ]:
import datetime
import time

import vertexai
from vertexai.generative_models import GenerativeModel
from vertexai.preview import caching

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Vertex AI 초기화 완료")

### 1-3. 대규모 시스템 프롬프트(캐시) 생성

약 **10,000 ~ 12,000 토큰** 분량의 더미 텍스트를 시스템 프롬프트로 사용하여 캐시를 생성합니다.  
이 캐시가 이후 모든 호출에서 재활용되며, TPM 할당량에 어떻게 반영되는지가 핵심 검증 포인트입니다.

In [ ]:
print("대규모 시스템 프롬프트(캐시) 생성 중...")

long_system_instruction = (
    "당신은 사내 보안 규정과 아키텍처를 안내하는 전문 AI 어시스턴트입니다. "
    * SYSTEM_PROMPT_REPEAT
)

print(f"시스템 프롬프트 길이: {len(long_system_instruction):,} 글자")
print(f"예상 토큰 수: 약 {len(long_system_instruction) // 4:,} 토큰 (추정)")

In [ ]:
cached_content = caching.CachedContent.create(
    model_name=MODEL_ID,
    system_instruction=long_system_instruction,
    ttl=datetime.timedelta(hours=CACHE_TTL_HOURS),
)

print(f"캐시 생성 완료!")
print(f"  캐시 이름: {cached_content.name}")
print(f"  캐시 만료: {CACHE_TTL_HOURS}시간 후")

### 1-4. 캐시를 활용한 10회 연속 API 호출

사내 개발자가 던질 법한 짧은 질문을 시뮬레이션하여 10회 연속 호출합니다.  
각 호출마다 `usageMetadata`에서 토큰 수치를 추출합니다.

#### 확인 포인트
- `prompt_token_count` (Total Input): 매 호출마다 **약 12,000대 이상**으로 나오는지 확인
- `cached_content_token_count`: 캐시된 토큰이 별도로 표기되는지 확인
- 이 `prompt_token_count` 전체가 Vertex AI가 인식하는 **1회 호출당 Input 토큰량**입니다

In [ ]:
print(f"캐시를 활용한 {NUM_CALLS}회 연속 API 호출 시작...\n")
print("=" * 65)

model = GenerativeModel.from_cached_content(cached_content=cached_content)

results = []

for i in range(1, NUM_CALLS + 1):
    response = model.generate_content("클라우드 보안 설정에 대해 요약해 줘.")

    usage = response.usage_metadata

    result = {
        "call_number": i,
        "prompt_token_count": usage.prompt_token_count,
        "cached_content_token_count": usage.cached_content_token_count,
        "candidates_token_count": usage.candidates_token_count,
    }
    results.append(result)

    print(f"--- [호출 {i:2d}/{NUM_CALLS}] ---")
    print(f"  Total Input  (prompt_token_count)          : {usage.prompt_token_count:>8,}")
    print(f"  └ Cached     (cached_content_token_count)  : {usage.cached_content_token_count:>8,}")
    print(f"  Output       (candidates_token_count)      : {usage.candidates_token_count:>8,}")
    print()

print("=" * 65)
print(f"{NUM_CALLS}회 호출 완료!")

### 1-5. 결과 요약 및 분석

In [ ]:
total_prompt_tokens = sum(r["prompt_token_count"] for r in results)
total_cached_tokens = sum(r["cached_content_token_count"] for r in results)
total_output_tokens = sum(r["candidates_token_count"] for r in results)
avg_prompt_per_call = total_prompt_tokens / len(results)

print("━" * 65)
print("                    결과 요약")
print("━" * 65)
print(f"  총 호출 횟수                          : {len(results):>10}회")
print(f"  1회 평균 Input 토큰 (prompt_token)    : {avg_prompt_per_call:>10,.0f}")
print(f"  총 Input 토큰 합계                    : {total_prompt_tokens:>10,}")
print(f"  총 Cached 토큰 합계                   : {total_cached_tokens:>10,}")
print(f"  총 Output 토큰 합계                   : {total_output_tokens:>10,}")
print("━" * 65)
print()
print("▶ 예상 TPM 차감량 (Input만):")
print(f"  prompt_token_count × {NUM_CALLS}회 = {total_prompt_tokens:,} 토큰")
print()
print("▶ 만약 캐시가 할인된다면 (Non-cached만 차감 시):")
non_cached_total = total_prompt_tokens - total_cached_tokens
print(f"  (prompt - cached) × {NUM_CALLS}회 = {non_cached_total:,} 토큰")
print()
print("━" * 65)
print("↑ GCP 콘솔 할당량 그래프에서 실제 TPM 피크가")
print(f"  {total_prompt_tokens:,}에 가까우면 → 캐시 토큰도 1:1로 TPM 차감")
print(f"  {non_cached_total:,}에 가까우면 → 캐시 토큰은 TPM에서 할인")
print("━" * 65)

### 1-6. 호출별 토큰 사용량 시각화

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker

    calls = [r["call_number"] for r in results]
    prompts = [r["prompt_token_count"] for r in results]
    cached = [r["cached_content_token_count"] for r in results]
    outputs = [r["candidates_token_count"] for r in results]

    fig, ax = plt.subplots(figsize=(12, 5))

    bar_width = 0.25
    x = range(len(calls))
    x1 = [i - bar_width for i in x]
    x2 = list(x)
    x3 = [i + bar_width for i in x]

    ax.bar(x1, prompts, width=bar_width, label="Total Input (prompt_token_count)", color="#4285F4")
    ax.bar(x2, cached, width=bar_width, label="Cached (cached_content_token_count)", color="#FBBC04")
    ax.bar(x3, outputs, width=bar_width, label="Output (candidates_token_count)", color="#34A853")

    ax.set_xlabel("호출 번호")
    ax.set_ylabel("토큰 수")
    ax.set_title("호출별 토큰 사용량 비교")
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"#{c}" for c in calls])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib가 설치되어 있지 않습니다.")
    print("시각화를 원하시면: !pip install matplotlib")

---
## 2단계: GCP 콘솔에서 실시간 할당량(TPM) 차감 그래프 확인

코드를 실행하여 트래픽 스파이크를 발생시켰으니, 이제 **할당량 엔진이 이를 어떻게 계산했는지** 눈으로 직접 확인할 차례입니다.  
(데이터 액세스 로그를 켤 필요 없이 기본 시스템 지표로 확인 가능합니다.)

### 콘솔 확인 절차

#### Step 1. GCP 콘솔 이동
- **IAM 및 관리자 > 할당량 및 시스템 한도** 페이지로 접속합니다.

#### Step 2. 필터 설정
상단 필터(Filter) 창에 다음 두 가지 조건을 넣습니다:

| 필터 항목 | 값 |
|-----------|----|
| **서비스** | `Vertex AI API` |
| **할당량** | `online_prediction_input_tokens_per_minute_per_base_model` |

#### Step 3. 그래프 확인
- 필터링된 목록에서 방금 테스트한 **리전(`us-central1`)**과 **모델명(`gemini-2.5-flash`)**에 해당하는 항목을 찾습니다.
- 우측의 **[지표 보기(View Metrics)]** 아이콘을 클릭하여 차트 창을 엽니다.

#### Step 4. 결과 분석 (중요)
- 그래프의 시간 범위를 **'최근 1시간'**으로 좁힙니다.  
  (참고: 지표가 그래프에 반영되기까지 **약 2~3분 정도의 지연**이 있을 수 있습니다.)
- 그래프에서 **스파이크가 튄 정점(Peak) 수치**를 확인합니다.

### 판정 기준

아래 셀에서 계산된 값과 GCP 콘솔 그래프의 피크 값을 비교하세요.

In [ ]:
print("━" * 65)
print("           2단계: GCP 콘솔 그래프 비교 기준값")
print("━" * 65)
print()
print("GCP 콘솔 할당량 그래프의 피크 값과 아래 수치를 비교하세요:")
print()
print(f"  [A] 캐시 포함 전체 Input 토큰 합계 : {total_prompt_tokens:>10,}")
print(f"  [B] 캐시 제외 Input 토큰 합계      : {non_cached_total:>10,}")
print()
print("─" * 65)
print("  판정:")
print(f"  • 그래프 피크 ≈ [A] {total_prompt_tokens:,}")
print(f"    → 캐시 할인 없음! TPM은 prompt_token_count 기준 1:1 차감")
print()
print(f"  • 그래프 피크 ≈ [B] {non_cached_total:,}")
print(f"    → 캐시 토큰은 TPM에서 할인됨")
print("─" * 65)
print()
print("⏳ 참고: 지표 반영까지 2~3분 지연이 있을 수 있습니다.")
print("   콘솔에서 시간 범위를 '최근 1시간'으로 설정하세요.")

---
## 정리: 캐시 삭제

테스트가 끝나면 불필요한 캐시를 삭제하여 비용을 절약합니다.  
TTL을 1시간으로 설정했으므로 자동 만료되지만, 즉시 삭제하려면 아래 셀을 실행하세요.

In [ ]:
try:
    cached_content.delete()
    print(f"캐시 삭제 완료: {cached_content.name}")
except Exception as e:
    print(f"캐시 삭제 중 오류 (이미 만료되었을 수 있음): {e}")

---
## 결론 기록

아래 셀에 테스트 결과를 기록하세요.

In [ ]:
# ────────────────────────────────────────────
#  테스트 결과 기록 (직접 작성)
# ────────────────────────────────────────────

test_result = {
    "테스트 일시": "YYYY-MM-DD HH:MM",
    "테스트 모델": MODEL_ID,
    "리전": LOCATION,
    "호출 횟수": NUM_CALLS,
    "1회 평균 prompt_token_count": "여기에 기록",
    "GCP 콘솔 피크 값": "여기에 기록",
    "판정": "캐시 할인 없음(1:1 차감) / 캐시 할인 적용",  # 택 1
    "비고": "",
}

print("테스트 결과:")
for k, v in test_result.items():
    print(f"  {k}: {v}")

---
## 부록: 게이트웨이 프록시 설계 시 참고사항

위 테스트 결과를 기반으로 게이트웨이 프록시(LiteLLM 등)의 할당량 제한 로직을 설계할 때:

### 캐시가 1:1로 TPM에 차감되는 경우 (예상 시나리오)

```
TPM 소모량 = prompt_token_count (캐시 포함 전체) + candidates_token_count
```

- 프록시에서 Rate Limiting 계산 시 `cached_content_token_count`를 빼면 **안 됩니다**.
- BigQuery 로그 적재 시 `prompt_token_count` 전체를 TPM 차감 기준으로 기록해야 합니다.

### 캐시가 할인되는 경우

```
TPM 소모량 = (prompt_token_count - cached_content_token_count) + candidates_token_count
```

- 프록시에서 캐시 토큰을 차감한 값으로 Rate Limiting을 계산할 수 있습니다.